In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

file_path = "/content/drive/MyDrive/MARG SIH/PAIMANA_feature_engineered_dataset.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (23115, 53)

Columns:
['report_file', 'format', 'sl_no', 'project_name', 'agency', 'project_code', 'state', 'ministry', 'sector', 'approval_date', 'revised_start_date', 'target_doc', 'revised_doc', 'original_cost_cr', 'revised_cost_cr', 'cumulative_expenditure_cr', 'physical_progress_pct', 'participating_states', 'is_multi_state', 'state_standardized', 'progress_invalid_flag', 'expenditure_invalid_flag', 'expenditure_above_original_flag', 'expenditure_above_revised_flag', 'approval_date_invalid_flag', 'revised_start_date_invalid_flag', 'target_doc_invalid_flag', 'revised_doc_invalid_flag', 'snapshot_date', 'project_name_normalized', 'project_id', 'prev_progress', 'prev_expenditure', 'prev_revised_cost', 'progress_growth', 'expenditure_growth', 'cost_growth', 'expenditure_vs_original_ratio', 'expenditure_vs_revised_ratio', 'months_since_prev_snapshot', 'progress_velocity', 'expenditure_velocity', 'cost_overrun_ratio', 'cost_overrun_cr', 'planned_duration_days', 'elapsed_duration_

/tmp/ipykernel_745/1487556058.py:5: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [4]:
cols = [
    "project_id",
    "snapshot_date",
    "original_cost_cr",
    "revised_cost_cr",
    "cumulative_expenditure_cr",
    "physical_progress_pct",
    "target_doc",
    "revised_doc",
    "cost_overrun_ratio",
    "planned_duration_days",
    "elapsed_duration_days",
    "time_elapsed_ratio",
    "progress_gap",
    "schedule_pressure"
]

print(df[cols].dtypes)

project_id                    object
snapshot_date                 object
original_cost_cr             float64
revised_cost_cr              float64
cumulative_expenditure_cr    float64
physical_progress_pct        float64
target_doc                    object
revised_doc                   object
cost_overrun_ratio           float64
planned_duration_days        float64
elapsed_duration_days        float64
time_elapsed_ratio           float64
progress_gap                 float64
schedule_pressure            float64
dtype: object


In [5]:
# Number of snapshots available for each project
snapshot_counts = df.groupby("project_id").size()

print("Projects with 1 snapshot :", (snapshot_counts == 1).sum())
print("Projects with 2+ snapshots:", (snapshot_counts >= 2).sum())
print("Projects with 3+ snapshots:", (snapshot_counts >= 3).sum())
print("Maximum snapshots:", snapshot_counts.max())

Projects with 1 snapshot : 1735
Projects with 2+ snapshots: 7929
Projects with 3+ snapshots: 2513
Maximum snapshots: 7


In [6]:
# Sort chronologically within each project
df = df.sort_values(["project_id", "snapshot_date"]).reset_index(drop=True)

# Look at changes from the next available snapshot
df["next_revised_cost"] = (
    df.groupby("project_id")["revised_cost_cr"].shift(-1)
)

df["next_planned_end"] = (
    df.groupby("project_id")["planned_end_date"].shift(-1)
)

df["next_snapshot_date"] = (
    df.groupby("project_id")["snapshot_date"].shift(-1)
)

print("Rows with a future snapshot:", df["next_snapshot_date"].notna().sum())

print("\nRevised cost changes:")
print(
    (df["next_revised_cost"] != df["revised_cost_cr"]).value_counts(dropna=False)
)

print("\nPlanned end-date changes:")
print(
    (df["next_planned_end"] != df["planned_end_date"]).value_counts(dropna=False)
)

Rows with a future snapshot: 13451

Revised cost changes:
True     11989
False    11126
Name: count, dtype: int64

Planned end-date changes:
True     14796
False     8319
Name: count, dtype: int64


In [7]:
# Future maximum revised cost for each project
df["future_max_revised_cost"] = (
    df.groupby("project_id")["revised_cost_cr"]
      .transform(lambda x: x.iloc[::-1].cummax().iloc[::-1])
)

# Future cost overrun target
df["future_cost_overrun"] = (
    df["future_max_revised_cost"] > df["original_cost_cr"]
)

print("Future cost overrun counts:")
print(df["future_cost_overrun"].value_counts(dropna=False))

Future cost overrun counts:
future_cost_overrun
False    17564
True      5551
Name: count, dtype: int64


In [8]:
# Maximum revised cost AFTER the current snapshot
df["future_max_revised_cost"] = (
    df.groupby("project_id")["revised_cost_cr"]
      .transform(
          lambda x: x.shift(-1)[::-1].cummax()[::-1]
      )
)

# 1 = project will exceed original cost in a future snapshot
# 0 = it will not
df["cost_overrun_target"] = (
    df["future_max_revised_cost"] > df["original_cost_cr"]
).astype("Int64")

print("Cost overrun target:")
print(df["cost_overrun_target"].value_counts(dropna=False))

Cost overrun target:
cost_overrun_target
0    19964
1     3151
Name: count, dtype: Int64


In [10]:
date_cols = [
    "snapshot_date",
    "target_doc",
    "revised_doc",
    "revised_start_date",
    "planned_end_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[date_cols].dtypes)

snapshot_date         datetime64[ns]
target_doc            datetime64[ns]
revised_doc           datetime64[ns]
revised_start_date    datetime64[ns]
planned_end_date      datetime64[ns]
dtype: object


In [11]:
df["future_max_planned_end"] = (
    df.groupby("project_id")["planned_end_date"]
      .transform(
          lambda x: x.shift(-1).iloc[::-1].cummax().iloc[::-1]
      )
)

df["time_overrun_target"] = (
    df["future_max_planned_end"] > df["planned_end_date"]
).astype("Int64")

print("Time overrun target:")
print(df["time_overrun_target"].value_counts(dropna=False))

Time overrun target:
time_overrun_target
0    21032
1     2083
Name: count, dtype: Int64


In [12]:
leakage_cols = [
    "cost_overrun_target",
    "time_overrun_target",
    "future_max_revised_cost",
    "future_max_planned_end",

    # Current/future outcome-related fields
    "revised_cost_cr",
    "cost_overrun_ratio",
    "cost_overrun_cr",

    # IDs / raw text / report metadata
    "project_id",
    "project_name",
    "project_name_normalized",
    "project_code",
    "report_file",
    "sl_no",

    # Administrative categorical fields for now
    "agency",
    "ministry",
    "sector",
    "state",
    "state_standardized",
    "participating_states",
    "format",

    # Raw date fields - we'll use engineered temporal features
    "approval_date",
    "revised_start_date",
    "target_doc",
    "revised_doc",
    "planned_end_date",
    "snapshot_date"
]

model_df = df.drop(columns=leakage_cols, errors="ignore").copy()

print("Original shape:", df.shape)
print("Model dataset shape:", model_df.shape)
print("\nModel features:")
print(model_df.columns.tolist())

Original shape: (23115, 61)
Model dataset shape: (23115, 35)

Model features:
['original_cost_cr', 'cumulative_expenditure_cr', 'physical_progress_pct', 'is_multi_state', 'progress_invalid_flag', 'expenditure_invalid_flag', 'expenditure_above_original_flag', 'expenditure_above_revised_flag', 'approval_date_invalid_flag', 'revised_start_date_invalid_flag', 'target_doc_invalid_flag', 'revised_doc_invalid_flag', 'prev_progress', 'prev_expenditure', 'prev_revised_cost', 'progress_growth', 'expenditure_growth', 'cost_growth', 'expenditure_vs_original_ratio', 'expenditure_vs_revised_ratio', 'months_since_prev_snapshot', 'progress_velocity', 'expenditure_velocity', 'planned_duration_days', 'elapsed_duration_days', 'time_elapsed_ratio', 'expected_progress_pct', 'progress_gap', 'schedule_pressure', 'remaining_progress_pct', 'expenditure_intensity', 'next_revised_cost', 'next_planned_end', 'next_snapshot_date', 'future_cost_overrun']


In [13]:
# Columns that must NEVER be used as ML inputs
common_exclude = [
    # Future information
    "next_revised_cost",
    "next_planned_end",
    "next_snapshot_date",
    "future_max_revised_cost",
    "future_max_planned_end",
    "future_cost_overrun",

    # Targets
    "cost_overrun_target",
    "time_overrun_target",

    # Identifiers / raw metadata
    "project_id",
    "project_name",
    "project_name_normalized",
    "project_code",
    "report_file",
    "sl_no",

    # Administrative categorical fields
    "agency",
    "ministry",
    "sector",
    "state",
    "state_standardized",
    "participating_states",
    "format",

    # Raw dates
    "approval_date",
    "revised_start_date",
    "target_doc",
    "revised_doc",
    "planned_end_date",
    "snapshot_date"
]

# Features that directly reveal current cost revision
cost_exclude = [
    "revised_cost_cr",
    "cost_overrun_ratio",
    "cost_overrun_cr",
    "expenditure_vs_revised_ratio"
]

# -------- COST MODEL --------
cost_features = [
    col for col in df.columns
    if col not in common_exclude + cost_exclude
]

X_cost = df[cost_features].copy()
y_cost = df["cost_overrun_target"].copy()


# -------- TIME MODEL --------
time_features = [
    col for col in df.columns
    if col not in common_exclude
]

X_time = df[time_features].copy()
y_time = df["time_overrun_target"].copy()


print("COST MODEL")
print("Features:", len(cost_features))
print(cost_features)

print("\nTIME MODEL")
print("Features:", len(time_features))
print(time_features)

COST MODEL
Features: 30
['original_cost_cr', 'cumulative_expenditure_cr', 'physical_progress_pct', 'is_multi_state', 'progress_invalid_flag', 'expenditure_invalid_flag', 'expenditure_above_original_flag', 'expenditure_above_revised_flag', 'approval_date_invalid_flag', 'revised_start_date_invalid_flag', 'target_doc_invalid_flag', 'revised_doc_invalid_flag', 'prev_progress', 'prev_expenditure', 'prev_revised_cost', 'progress_growth', 'expenditure_growth', 'cost_growth', 'expenditure_vs_original_ratio', 'months_since_prev_snapshot', 'progress_velocity', 'expenditure_velocity', 'planned_duration_days', 'elapsed_duration_days', 'time_elapsed_ratio', 'expected_progress_pct', 'progress_gap', 'schedule_pressure', 'remaining_progress_pct', 'expenditure_intensity']

TIME MODEL
Features: 34
['original_cost_cr', 'revised_cost_cr', 'cumulative_expenditure_cr', 'physical_progress_pct', 'is_multi_state', 'progress_invalid_flag', 'expenditure_invalid_flag', 'expenditure_above_original_flag', 'expendit

In [14]:
print(
    df["snapshot_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

snapshot_date
2025-04    1670
2025-05    1637
2025-06    1595
2025-07     714
2025-08     800
2025-09     772
2025-10     798
2025-11     823
2025-12    1345
2026-01    1605
2026-02    1897
2026-03    1869
2026-04    1981
2026-05    1987
2026-06    1847
2026-07    1775
Freq: M, Name: count, dtype: int64


In [15]:
print("First snapshot:", df["snapshot_date"].min())
print("Last snapshot :", df["snapshot_date"].max())

First snapshot: 2025-04-01 00:00:00
Last snapshot : 2026-07-01 00:00:00


In [16]:
train_mask = df["snapshot_date"] < "2026-04-01"
test_mask  = df["snapshot_date"] >= "2026-04-01"

print("Training rows:", train_mask.sum())
print("Testing rows :", test_mask.sum())

print("\nTraining period:")
print(df.loc[train_mask, "snapshot_date"].min(),
      "to",
      df.loc[train_mask, "snapshot_date"].max())

print("\nTesting period:")
print(df.loc[test_mask, "snapshot_date"].min(),
      "to",
      df.loc[test_mask, "snapshot_date"].max())

Training rows: 15525
Testing rows : 7590

Training period:
2025-04-01 00:00:00 to 2026-03-01 00:00:00

Testing period:
2026-04-01 00:00:00 to 2026-07-01 00:00:00


In [17]:
print("COST TARGET")
print("Train:")
print(y_cost[train_mask].value_counts(dropna=False))
print("\nTest:")
print(y_cost[test_mask].value_counts(dropna=False))

print("\nTIME TARGET")
print("Train:")
print(y_time[train_mask].value_counts(dropna=False))
print("\nTest:")
print(y_time[test_mask].value_counts(dropna=False))

COST TARGET
Train:
cost_overrun_target
0    13340
1     2185
Name: count, dtype: Int64

Test:
cost_overrun_target
0    6624
1     966
Name: count, dtype: Int64

TIME TARGET
Train:
time_overrun_target
0    13989
1     1536
Name: count, dtype: Int64

Test:
time_overrun_target
0    7043
1     547
Name: count, dtype: Int64


In [18]:
# =========================
# COST MODEL DATA
# =========================

cost_train_valid = train_mask & y_cost.notna()
cost_test_valid  = test_mask & y_cost.notna()

X_cost_train = X_cost.loc[cost_train_valid].copy()
y_cost_train = y_cost.loc[cost_train_valid].astype(int)

X_cost_test = X_cost.loc[cost_test_valid].copy()
y_cost_test = y_cost.loc[cost_test_valid].astype(int)


# =========================
# TIME MODEL DATA
# =========================

time_train_valid = train_mask & y_time.notna()
time_test_valid  = test_mask & y_time.notna()

X_time_train = X_time.loc[time_train_valid].copy()
y_time_train = y_time.loc[time_train_valid].astype(int)

X_time_test = X_time.loc[time_test_valid].copy()
y_time_test = y_time.loc[time_test_valid].astype(int)


print("COST MODEL")
print("X_train:", X_cost_train.shape)
print("y_train:", y_cost_train.shape)
print("X_test :", X_cost_test.shape)
print("y_test :", y_cost_test.shape)

print("\nTIME MODEL")
print("X_train:", X_time_train.shape)
print("y_train:", y_time_train.shape)
print("X_test :", X_time_test.shape)
print("y_test :", y_time_test.shape)

COST MODEL
X_train: (15525, 30)
y_train: (15525,)
X_test : (7590, 30)
y_test : (7590,)

TIME MODEL
X_train: (15525, 34)
y_train: (15525,)
X_test : (7590, 34)
y_test : (7590,)


In [19]:
print("COST feature types:")
print(X_cost_train.dtypes.value_counts())

print("\nTIME feature types:")
print(X_time_train.dtypes.value_counts())

COST feature types:
float64    21
bool        9
Name: count, dtype: int64

TIME feature types:
float64    25
bool        9
Name: count, dtype: int64


In [20]:
print("Non-numeric COST columns:")
print(X_cost_train.select_dtypes(exclude="number").columns.tolist())

print("\nNon-numeric TIME columns:")
print(X_time_train.select_dtypes(exclude="number").columns.tolist())

Non-numeric COST columns:
['is_multi_state', 'progress_invalid_flag', 'expenditure_invalid_flag', 'expenditure_above_original_flag', 'expenditure_above_revised_flag', 'approval_date_invalid_flag', 'revised_start_date_invalid_flag', 'target_doc_invalid_flag', 'revised_doc_invalid_flag']

Non-numeric TIME columns:
['is_multi_state', 'progress_invalid_flag', 'expenditure_invalid_flag', 'expenditure_above_original_flag', 'expenditure_above_revised_flag', 'approval_date_invalid_flag', 'revised_start_date_invalid_flag', 'target_doc_invalid_flag', 'revised_doc_invalid_flag']


In [21]:
!pip -q install lightgbm

import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [22]:
cost_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    class_weight="balanced"
)

cost_model.fit(X_cost_train, y_cost_train)

[LightGBM] [Info] Number of positive: 2185, number of negative: 13340
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001558 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4885
[LightGBM] [Info] Number of data points in the train set: 15525, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


LGBMClassifier(class_weight='balanced', learning_rate=0.05, n_estimators=300,
               objective='binary', random_state=42)

In [23]:
cost_pred = cost_model.predict(X_cost_test)
cost_prob = cost_model.predict_proba(X_cost_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_cost_test, cost_pred), 3))
print("Precision:", round(precision_score(y_cost_test, cost_pred), 3))
print("Recall   :", round(recall_score(y_cost_test, cost_pred), 3))
print("F1 Score :", round(f1_score(y_cost_test, cost_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_cost_test, cost_prob), 3))

print("\nConfusion Matrix")
print(confusion_matrix(y_cost_test, cost_pred))

Accuracy : 0.806
Precision: 0.37
Recall   : 0.744
F1 Score : 0.494
ROC-AUC  : 0.853

Confusion Matrix
[[5398 1226]
 [ 247  719]]


In [24]:
importance = (
    pd.DataFrame({
        "Feature": X_cost_train.columns,
        "Importance": cost_model.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

importance.head(10)

,Feature,Importance
0,original_cost_cr,1292
23,elapsed_duration_days,936
29,expenditure_intensity,887
14,prev_revised_cost,745
1,cumulative_expenditure_cr,646
18,expenditure_vs_original_ratio,633
22,planned_duration_days,596
24,time_elapsed_ratio,571
2,physical_progress_pct,467
19,months_since_prev_snapshot,352


In [26]:
# Correct target labels: projects with no future outcome should be unlabeled
import numpy as np
valid_cost = (
    df["future_max_revised_cost"].notna()
    & df["original_cost_cr"].notna()
)

df["cost_overrun_target"] = pd.Series(
    np.where(
        valid_cost,
        (df["future_max_revised_cost"] > df["original_cost_cr"]).astype(int),
        np.nan
    ),
    index=df.index
).astype("Int64")


valid_time = (
    df["future_max_planned_end"].notna()
    & df["planned_end_date"].notna()
)

df["time_overrun_target"] = pd.Series(
    np.where(
        valid_time,
        (df["future_max_planned_end"] > df["planned_end_date"]).astype(int),
        np.nan
    ),
    index=df.index
).astype("Int64")


print("COST TARGET")
print(df["cost_overrun_target"].value_counts(dropna=False))

print("\nTIME TARGET")
print(df["time_overrun_target"].value_counts(dropna=False))

COST TARGET
cost_overrun_target
<NA>    11837
0        8127
1        3151
Name: count, dtype: Int64

TIME TARGET
time_overrun_target
<NA>    13034
0        7998
1        2083
Name: count, dtype: Int64


In [27]:
# Rebuild train/test sets using only rows with known future outcomes

cost_train_valid = (
    (df["snapshot_date"] < "2026-04-01")
    & df["cost_overrun_target"].notna()
)

cost_test_valid = (
    (df["snapshot_date"] >= "2026-04-01")
    & df["cost_overrun_target"].notna()
)

time_train_valid = (
    (df["snapshot_date"] < "2026-04-01")
    & df["time_overrun_target"].notna()
)

time_test_valid = (
    (df["snapshot_date"] >= "2026-04-01")
    & df["time_overrun_target"].notna()
)

# Cost
X_cost_train = X_cost.loc[cost_train_valid].copy()
y_cost_train = df.loc[cost_train_valid, "cost_overrun_target"].astype(int)

X_cost_test = X_cost.loc[cost_test_valid].copy()
y_cost_test = df.loc[cost_test_valid, "cost_overrun_target"].astype(int)

# Time
X_time_train = X_time.loc[time_train_valid].copy()
y_time_train = df.loc[time_train_valid, "time_overrun_target"].astype(int)

X_time_test = X_time.loc[time_test_valid].copy()
y_time_test = df.loc[time_test_valid, "time_overrun_target"].astype(int)

print("COST")
print("Train:", X_cost_train.shape, y_cost_train.value_counts().to_dict())
print("Test :", X_cost_test.shape, y_cost_test.value_counts().to_dict())

print("\nTIME")
print("Train:", X_time_train.shape, y_time_train.value_counts().to_dict())
print("Test :", X_time_test.shape, y_time_test.value_counts().to_dict())

COST
Train: (7428, 30) {0: 5243, 1: 2185}
Test : (3850, 30) {0: 2884, 1: 966}

TIME
Train: (7358, 34) {0: 5822, 1: 1536}
Test : (2723, 34) {0: 2176, 1: 547}


In [28]:
cost_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    class_weight="balanced"
)

cost_model.fit(X_cost_train, y_cost_train)

print("Cost model trained successfully.")

[LightGBM] [Info] Number of positive: 2185, number of negative: 5243
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000732 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4358
[LightGBM] [Info] Number of data points in the train set: 7428, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Cost model trained successfully.


In [29]:
cost_pred = cost_model.predict(X_cost_test)
cost_prob = cost_model.predict_proba(X_cost_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_cost_test, cost_pred), 3))
print("Precision:", round(precision_score(y_cost_test, cost_pred), 3))
print("Recall   :", round(recall_score(y_cost_test, cost_pred), 3))
print("F1 Score :", round(f1_score(y_cost_test, cost_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_cost_test, cost_prob), 3))

print("\nConfusion Matrix")
print(confusion_matrix(y_cost_test, cost_pred))

Accuracy : 0.844
Precision: 0.706
Recall   : 0.648
F1 Score : 0.676
ROC-AUC  : 0.89

Confusion Matrix
[[2623  261]
 [ 340  626]]


In [30]:
time_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    class_weight="balanced"
)

time_model.fit(X_time_train, y_time_train)

print("Time model trained successfully.")

[LightGBM] [Info] Number of positive: 1536, number of negative: 5822
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001088 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5824
[LightGBM] [Info] Number of data points in the train set: 7358, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Time model trained successfully.


In [31]:
time_pred = time_model.predict(X_time_test)
time_prob = time_model.predict_proba(X_time_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_time_test, time_pred), 3))
print("Precision:", round(precision_score(y_time_test, time_pred), 3))
print("Recall   :", round(recall_score(y_time_test, time_pred), 3))
print("F1 Score :", round(f1_score(y_time_test, time_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_time_test, time_prob), 3))

print("\nConfusion Matrix")
print(confusion_matrix(y_time_test, time_pred))

Accuracy : 0.642
Precision: 0.336
Recall   : 0.803
F1 Score : 0.474
ROC-AUC  : 0.76

Confusion Matrix
[[1308  868]
 [ 108  439]]


In [32]:
time_importance = (
    pd.DataFrame({
        "Feature": X_time_train.columns,
        "Importance": time_model.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

print(time_importance.head(15).to_string(index=False))

                      Feature  Importance
        planned_duration_days         818
        expenditure_intensity         778
        elapsed_duration_days         657
             original_cost_cr         645
           time_elapsed_ratio         621
 expenditure_vs_revised_ratio         594
              revised_cost_cr         587
    cumulative_expenditure_cr         528
                 progress_gap         461
        physical_progress_pct         454
           cost_overrun_ratio         422
expenditure_vs_original_ratio         421
                prev_progress         299
              cost_overrun_cr         254
        expected_progress_pct         233


In [33]:
import joblib

joblib.dump(cost_model, "cost_overrun_model.pkl")
joblib.dump(time_model, "time_overrun_model.pkl")

joblib.dump(cost_features, "cost_features.pkl")
joblib.dump(time_features, "time_features.pkl")

print("Models and feature lists saved successfully.")

Models and feature lists saved successfully.


In [34]:
import os
import shutil

os.makedirs("M2_model_package", exist_ok=True)

shutil.copy("cost_overrun_model.pkl", "M2_model_package/cost_overrun_model.pkl")
shutil.copy("time_overrun_model.pkl", "M2_model_package/time_overrun_model.pkl")
shutil.copy("cost_features.pkl", "M2_model_package/cost_features.pkl")
shutil.copy("time_features.pkl", "M2_model_package/time_features.pkl")

print("M2 model package contents:")
print(os.listdir("M2_model_package"))

M2 model package contents:
['cost_features.pkl', 'cost_overrun_model.pkl', 'time_overrun_model.pkl', 'time_features.pkl']


In [35]:
import shutil

zip_path = shutil.make_archive(
    "M2_model_package",
    "zip",
    "M2_model_package"
)

print("Created:", zip_path)

Created: /content/M2_model_package.zip


In [36]:
from google.colab import files

files.download("M2_model_package.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>